In [3]:
"""
Minimal on-disk ANN index wrapper around USearch.

    pip install usearch numpy

Usage:
    store = AnnStore("my_index.usearch", dim=128)   # loads if the file exists
    store.insert(42, my_int8_array)
    store.save()
    keys, dists = store.search(query_array, k=5)
"""
import json
import os

import numpy as np
from usearch.index import Index


class AnnStore:
    def __init__(self, path, dim, metric="l2sq", dtype="i8",
                 connectivity=16, expansion_add=128, expansion_search=64):
        # dtype: "i8" (int8), "f16", "f32", or "b1" (bit-packed, use metric="hamming")
        self.path = path
        self.meta_path = path + ".meta.json"
        self.dim = dim
        self.dtype = dtype
        self.np_dtype = {"i8": np.int8, "f16": np.float16, "f32": np.float32}.get(dtype)

        self.index = Index(
            ndim=dim,
            metric=metric,
            dtype=dtype,
            connectivity=connectivity,
            expansion_add=expansion_add,
            expansion_search=expansion_search,
        )

        if os.path.exists(path):
            self._check_manifest()
            self.index.load(path)  # use self.index.view(path) to memory-map instead

    # ---------- persistence ----------
    def save(self):
        tmp = self.path + ".tmp"
        self.index.save(tmp)
        os.replace(tmp, self.path)  # atomic: a crash never leaves a half-written file
        with open(self.meta_path, "w") as f:
            json.dump({"dim": self.dim, "dtype": self.dtype, "count": len(self.index)}, f)

    def _check_manifest(self):
        if os.path.exists(self.meta_path):
            with open(self.meta_path) as f:
                meta = json.load(f)
            if meta["dim"] != self.dim or meta["dtype"] != self.dtype:
                raise ValueError(
                    f"Index on disk has dim={meta['dim']}, dtype={meta['dtype']}; "
                    f"requested dim={self.dim}, dtype={self.dtype}. Rebuild the index."
                )

    # ---------- operations ----------
    def _prep(self, arr):
        arr = np.asarray(arr)
        if self.np_dtype is not None:
            arr = arr.astype(self.np_dtype, copy=False)
        if arr.shape[-1] != self.dim:
            raise ValueError(f"Expected dimension {self.dim}, got {arr.shape[-1]}")
        return arr

    def insert(self, key, array):
        """Insert one vector under an integer key."""
        self.index.add(int(key), self._prep(array))

    def insert_many(self, keys, arrays):
        """Insert a batch (2D array, one row per key). Much faster than looping."""
        self.index.add(np.asarray(keys, dtype=np.uint64), self._prep(arrays))

    def search(self, array, k=10):
        """Return (keys, distances) of the k nearest neighbours of one vector."""
        matches = self.index.search(self._prep(array), k)
        return matches.keys, matches.distances

    def remove(self, key):
        """Mark a key as deleted (space is only reclaimed by rebuilding)."""
        self.index.remove(int(key))

    def __len__(self):
        return len(self.index)


if __name__ == "__main__":
    rng = np.random.default_rng(0)
    data = rng.integers(-128, 127, size=(1000, 64), dtype=np.int8)

    store = AnnStore("demo.usearch", dim=64)
    if len(store) == 0:
        store.insert_many(np.arange(1000), data)
        store.save()
        print("Built and saved index with", len(store), "vectors")
    else:
        print("Loaded index with", len(store), "vectors")

    keys, dists = store.search(data[123], k=3)
    print("Nearest keys:", keys, "distances:", dists)  # key 123 should come first

Built and saved index with 1000 vectors
Nearest keys: [123 239 563] distances: [     0. 340449. 362895.]


In [2]:
pip install usearch numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import requests

LAT_MIN, LAT_MAX = 61.40, 61.55
LON_MIN, LON_MAX = 23.60, 23.95


def normalize(lat: float, lon: float, size: int = 1000) -> tuple[int, int]:
    """Return (x, y) as ints: x = west->east, y = south->north, both in 0..size."""
    x = (lon - LON_MIN) / (LON_MAX - LON_MIN) * size
    y = (lat - LAT_MIN) / (LAT_MAX - LAT_MIN) * size
    x = max(0.0, min(float(size), x))
    y = max(0.0, min(float(size), y))
    return round(x), round(y)


def place_to_xy(query: str, size: int = 1000) -> tuple[int, int] | None:
    resp = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": query, "format": "json", "limit": 1},
        headers={"User-Agent": "hackathon/0.1 (ivars.nya@gmail.com)"},
        timeout=10,
    )
    resp.raise_for_status()
    results: list[dict[str, str]] = resp.json()
    if not results:
        return None
    return normalize(float(results[0]["lat"]), float(results[0]["lon"]), size)

if __name__ == "__main__":
    print(place_to_xy("Tampere Central Square"))

(460, 653)


In [16]:
"""
Category table: category (str) -> options (str[]) and dimensions (int[]).

options[i] corresponds to dimensions[i]. Each option owns one vector dimension.
The global nextDim is the lowest dimension not yet used.
"""
import json
import os

CATCHALL = "factors within this category that aren't covered by any of the options"

# category -> {"options": [str], "dimensions": [int]}
table = {}
nextDim = 0


def _allocate(count):
    """Hand out `count` consecutive dimensions starting at nextDim."""
    global nextDim
    dims = list(range(nextDim, nextDim + count))
    nextDim += count
    return dims


def add_category(category, options):
    """Create a new category. The catch-all option is always placed first."""
    if category in table:
        raise ValueError(f"Category '{category}' already exists")
    options = [o for o in options if o != CATCHALL]  # avoid duplicating the catch-all
    if len(set(options)) != len(options):
        raise ValueError("Duplicate options given")
    all_options = [CATCHALL] + options
    table[category] = {"options": all_options, "dimensions": _allocate(len(all_options))}


def add_options(category, options):
    """Append options to an existing category, using dimensions from nextDim upward."""
    if category not in table:
        raise KeyError(f"Category '{category}' does not exist")
    entry = table[category]
    new = [o for o in options if o not in entry["options"]]  # skip ones already present
    if len(set(new)) != len(new):
        raise ValueError("Duplicate options given")
    entry["options"].extend(new)
    entry["dimensions"].extend(_allocate(len(new)))


def save(path="categories.json"):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(table, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)  # atomic


def load(path="categories.json"):
    """Load the table and recompute nextDim as (highest used dimension) + 1."""
    global table, nextDim
    if not os.path.exists(path):
        table, nextDim = {}, 0
        return
    with open(path, encoding="utf-8") as f:
        table = json.load(f)
    used = [d for entry in table.values() for d in entry["dimensions"]]
    nextDim = max(used) + 1 if used else 0


if __name__ == "__main__":
    load("demo_categories.json")
    if not table:
        add_category("colour", ["red", "green"])
        add_category("size", ["small", "large"])
        add_options("colour", ["blue"])
        save("demo_categories.json")
    for cat, entry in table.items():
        print(cat, list(zip(entry["dimensions"], entry["options"])))
    print("nextDim =", nextDim)

colour [(0, "factors within this category that aren't covered by any of the options"), (1, 'red'), (2, 'green'), (6, 'blue')]
size [(3, "factors within this category that aren't covered by any of the options"), (4, 'small'), (5, 'large')]
nextDim = 7


In [2]:
"""Load the tower table and look up (x, y) by code, name, or cell ID.

    from tower_lookup import TowerIndex
    towers = TowerIndex("towers.csv.gz")
    towers.get("FI-KAL-M00001")      # -> (412, 733) or None
    towers["Kalamari Centre Mast 1"] # -> (x, y), raises KeyError if missing
    towers.get(100000)               # numeric cell ID works too

Lookups are case-insensitive and ignore surrounding whitespace.
"""
import csv
import gzip


def _key(value):
    return str(value).strip().casefold()


class TowerIndex:
    def __init__(self, path=None):
        self._by_key = {}
        self._rows = 0
        if path:
            self.load(path)

    def load(self, path):
        opener = gzip.open if path.endswith(".gz") else open
        with opener(path, "rt", newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                xy = (int(row["x"]), int(row["y"]))
                # one dict, three kinds of key: code, name, cell_id
                self._by_key[_key(row["code"])] = xy
                self._by_key[_key(row["name"])] = xy
                self._by_key[_key(row["cell_id"])] = xy
                self._rows += 1
        return self

    def get(self, query, default=None):
        return self._by_key.get(_key(query), default)

    def __getitem__(self, query):
        return self._by_key[_key(query)]

    def __contains__(self, query):
        return _key(query) in self._by_key

    def __len__(self):
        """Number of sites (rows loaded)."""
        return self._rows


def _in_ipython():
    try:
        get_ipython  # defined only inside IPython/Jupyter
        return True
    except NameError:
        return False


if __name__ == "__main__" and not _in_ipython():
    import sys

    if len(sys.argv) < 3:
        sys.exit("usage: python tower_lookup.py TABLE.csv[.gz] QUERY [QUERY ...]")
    idx = TowerIndex(sys.argv[1])
    print(f"Loaded {len(idx)} sites")
    for q in sys.argv[2:]:
        print(q, "->", idx.get(q))

In [7]:
towers = TowerIndex("towers.csv.gz")
print(len(towers))                    # 200000
print(towers.get("FI-PRK-M00001"))    # (96, 362)

200000
(96, 362)


In [18]:
GEMINI_API_KEY=open("gemkey.txt", encoding="utf-8").read()

In [13]:
pip install google-genai

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.3 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 2.5 MB/s  0:00:00

   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---- ----------------------------------- 1/9 [truststore]
   -------- ------------------------------- 2/9 [tenacity]
  Attempting uninstall: idna
   -------- ------------------------------- 2/9 [tenacity]
    Found existing installation: idna 3.10
   -------- ------------------------------- 2/9 [tenacity]
    Uninstalling idna-3.10:
   -----


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
"""Thin wrapper around the Gemini API (google-genai SDK).

Set GEMINI_API_KEY in the environment (or server/.env) before calling.

    from gemini import ask_gemini
    print(ask_gemini("Say hi"))
"""

import os
from functools import lru_cache
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

#load_dotenv(Path(__file__).resolve().parent / ".env")

DEFAULT_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
# Tried in order when the main model is overloaded, times out or is unavailable.
FALLBACK_MODELS = [
    m.strip()
    for m in os.getenv(
        "GEMINI_FALLBACK_MODELS", "gemini-3.1-flash-lite"
    ).split(",")
    if m.strip()
]
TIMEOUT_SECONDS = float(os.getenv("GEMINI_TIMEOUT", "30"))


@lru_cache(maxsize=1)
def _client() -> genai.Client:
    api_key = GEMINI_API_KEY
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY is not set")
    return genai.Client(
        api_key=api_key,
        http_options=types.HttpOptions(timeout=int(TIMEOUT_SECONDS * 1000)),
    )


def ask_gemini(
    prompt: str,
    *,
    system: str | None = None,
    model: str = DEFAULT_MODEL,
    temperature: float | None = None,
    json_output: bool = False,
) -> str:
    """Send a prompt to Gemini and return the response text.

    system:      optional system instruction.
    json_output: ask the model to reply with JSON (parse the result with json.loads).
    """
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if json_output else None,
    )
    last_error: Exception | None = None
    for candidate in [model, *(m for m in FALLBACK_MODELS if m != model)]:
        try:
            response = _client().models.generate_content(
                model=candidate, contents=prompt, config=config
            )
            return response.text or ""
        except errors.ClientError as e:
            # Bad key / bad request won't be fixed by another model; 404 and 429 might.
            if e.code not in (404, 429):
                raise
            last_error = e
        except Exception as e:  # 5xx overload, timeouts, network errors
            last_error = e
        print(f"[gemini] {candidate} failed ({type(last_error).__name__}), trying next model")
    raise RuntimeError("All Gemini models failed") from last_error


if __name__ == "__main__":
    print(ask_gemini("Reply with one short sentence confirming you work."))

System operational and ready.


In [21]:
"""
Situations table: integer key, timestamp, vector (int8 array).

Stored in SQLite, so persistence is automatic: there is no separate save/load step.
The integer key is also the key you give to the ANN index, so the two stay linked.
"""
import sqlite3
from datetime import datetime, timezone

import numpy as np


class Situations:
    def __init__(self, path="situations.db", dim=64, dtype=np.int8):
        self.dim = dim
        self.dtype = np.dtype(dtype)
        self.db = sqlite3.connect(path)
        self.db.execute(
            """CREATE TABLE IF NOT EXISTS situations (
                   id        INTEGER PRIMARY KEY AUTOINCREMENT,
                   timestamp TEXT NOT NULL,   -- ISO 8601, UTC
                   vector    BLOB NOT NULL    -- dim * itemsize raw bytes
               )"""
        )
        self.db.commit()

    # ---------- helpers ----------
    def _to_blob(self, vector):
        arr = np.asarray(vector)
        if arr.shape != (self.dim,):
            raise ValueError(f"Expected shape ({self.dim},), got {arr.shape}")
        info = np.iinfo(self.dtype)
        if arr.min() < info.min or arr.max() > info.max:
            raise ValueError(f"Values must fit in {self.dtype} ({info.min}..{info.max})")
        return arr.astype(self.dtype).tobytes()

    def _from_blob(self, blob):
        return np.frombuffer(blob, dtype=self.dtype).copy()

    # ---------- operations ----------
    def add(self, vector, timestamp=None):
        """Insert a situation and return its new integer key."""
        if timestamp is None:
            timestamp = datetime.now(timezone.utc)
        ts = timestamp.isoformat() if isinstance(timestamp, datetime) else str(timestamp)
        cur = self.db.execute(
            "INSERT INTO situations (timestamp, vector) VALUES (?, ?)",
            (ts, self._to_blob(vector)),
        )
        self.db.commit()
        return cur.lastrowid

    def get(self, key):
        """Return (key, timestamp, vector) or None if the key doesn't exist."""
        row = self.db.execute(
            "SELECT id, timestamp, vector FROM situations WHERE id = ?", (key,)
        ).fetchone()
        return None if row is None else (row[0], row[1], self._from_blob(row[2]))

    def all(self):
        """Yield (key, timestamp, vector) for every row, e.g. to rebuild the ANN index."""
        for key, ts, blob in self.db.execute(
            "SELECT id, timestamp, vector FROM situations ORDER BY id"
        ):
            yield key, ts, self._from_blob(blob)

    def delete(self, key):
        self.db.execute("DELETE FROM situations WHERE id = ?", (key,))
        self.db.commit()

    def __len__(self):
        return self.db.execute("SELECT COUNT(*) FROM situations").fetchone()[0]

    def close(self):
        self.db.close()


if __name__ == "__main__":
    s = Situations("demo_situations.db", dim=8)
    k = s.add(np.array([0, 1, 0, 0, -3, 0, 2, 0]))
    print("Inserted key", k, "| total rows:", len(s))
    print(s.get(k))
    s.close()

Inserted key 1 | total rows: 1
(1, '2026-11-07T19:01:44.244597+00:00', array([ 0,  1,  0,  0, -3,  0,  2,  0], dtype=int8))


In [22]:
def get_categories():
    """Return every category name, in the order they were added."""
    return list(table)
 
 
def get_options(category, with_dimensions=False):
    """
    Return every option of a category (the catch-all first), as a new list.
    With with_dimensions=True, return (option, dimension) pairs instead.
    """
    if category not in table:
        raise KeyError(f"Category '{category}' does not exist")
    entry = table[category]
    if with_dimensions:
        return list(zip(entry["options"], entry["dimensions"]))
    return list(entry["options"])

In [25]:
import os

TEXT_DIR = "situation_texts"

def text_path(key):
    return os.path.join(TEXT_DIR, f"{int(key)}.txt")

def save_text(key, text):
    os.makedirs(TEXT_DIR, exist_ok=True)
    path = text_path(key)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        f.write(text)
    os.replace(tmp, path)          # atomic: no half-written text files

def load_text(key):
    with open(text_path(key), encoding="utf-8", newline="") as f:
        return f.read()            # raises FileNotFoundError if missing

def delete_text(key):
    try:
        os.remove(text_path(key))
    except FileNotFoundError:
        pass

def add_situation(situations, text):
    key = situations.add()         # SQLite creates the key first...
    save_text(key, text)           # ...then the text goes in <key>.txt
    return key

In [ ]:
import numpy as np
import re

COORD_RE = re.compile(
    r"^\s*[\(\[]?\s*([-+]?\d*\.?\d+)\s*(?:,|\s)\s*([-+]?\d*\.?\d+)\s*[\)\]]?\s*$"
)

def first_vectorization(text, categories):
    prompt = (
        "Read attachment 1 and return a location if discernible. Prioritize "
        "coordinates or a specific hardware location such as a cell tower; "
        "otherwise name the general location (city, or part thereof) with as "
        "much precision as you are confident in. Also return a score from 0 to "
        "1000 for how much information about the situation can't be represented "
        "in the categories in attachment 2.\n"
        'Respond with JSON only, in the form ["<location>", <score>].\n\n'
        "--- attachment 1 ---\n"
        f"{text}\n"
        "--- end attachment 1 ---\n\n"
        "--- attachment 2 ---\n"
        f"{', '.join(categories)}\n"
        "--- end attachment 2 ---"
    )

    data = json.loads(ask_gemini(prompt))
    location, score = str(data[0]), int(data[1])

    m = COORD_RE.match(location)
    if m:
        xy = normalize((float(m.group(1)), float(m.group(2))))
    else:
        xy = towers.get(location)
        if xy is None:
            xy = place_to_xy(location)

    return score, int(xy[0]), int(xy[1])


def general_vectorization(text, category, options):
    prompt = (
            "Within the category of {attachement 1} score each of the options from 0 to 1000 on how important they are to the situation in {attachement 2}. It is possible that none of them are important, if so, all scores should be low. Return only comma separated integers."
            "\n--- attachment 1 ---\n"
            f"{category}\n"
            f"{', '.join(options)}\n"
            "--- end attachment 1 ---\n\n"
            "--- attachment 2 ---\n"
            f"{text}\n"
            "--- end attachment 2 ---"
       )
    return json.loads(ask_gemini(prompt))


def _apply(vector, contribution, dims=None):
    """Add a stub's result (dict or array) into the working vector, in place.

    If dims is given, contribution[i] is added to vector[dims[i]].
    """
    if isinstance(contribution, dict):
        for dim, value in contribution.items():
            vector[dim] += value
    elif dims is not None:
        dims = np.asarray(dims, dtype=np.intp)
        values = np.asarray(contribution, dtype=np.int32)
        np.add.at(vector, dims, values)
    else:
        vector += np.asarray(contribution, dtype=np.int32)


def vectorize_situations(situations, store):
    categories = {c: ct.get_options(c, with_dimensions=True) for c in ct.get_categories()}
    info = np.iinfo(situations.dtype)
    done = 0

    for key in situations.pending():              # situations whose vector is NULL
        try:
            text = load_text(key)
        except FileNotFoundError:
            print(f"Skipping situation {key}: no text file")
            continue

        vector = np.zeros(situations.dim, dtype=np.int32)

        # 1) first stub: text + all categories
        contribution = first_vectorization(text, categories)
        _apply(vector[:3], contribution)

        # 2) second stub: text + one category + its options, once per category
        for category, (options, dims) in categories.items():
            contribution = general_vectorization(text, category, options)
            _apply(vector, contribution, dims=dims)

        vector = np.clip(vector, info.min, info.max).astype(situations.dtype)
        situations.set_vector(key, vector)        # set the vector in the table
        store.insert(key, vector)                 # and add it to the index
        done += 1

    if done:
        store.save()
    return done

IndentationError: unexpected indent (3183212915.py, line 53)